# 01. Preparar los datos de mercado

Este cuaderno recoge las principales fases de preparación: leer y limpiar las cotizaciones, incorporar las acciones en circulación, calcular variables de mercado, seleccionar empresas con una capitalización de **2 a 300 millones de dólares** e identificar los gaps.

**Datos de entrada:** `barras_diarias_2021_2025.csv`, `barras_diarias_2026.csv`, `mapa_ticker_cik.csv`, `acciones_circulacion_sec.csv` y `universo.csv`. Proceden de Alpaca y la SEC. El script opcional `descargar_datos.py` documenta la descarga.

## 1. Librerías y ficheros

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Carpeta con los datos descargados
DATOS = Path("data")

# Archivos de precios
fichero_precios = DATOS / "barras_diarias_2021_2025.csv"
fichero_2026 = DATOS / "barras_diarias_2026.csv"

# Cargamos los datos de ambos periodos
precios = pd.read_csv(fichero_precios)
precios_2026 = pd.read_csv(fichero_2026)

print("Sesiones 2021-2025:", len(precios))
print("Sesiones 2026:", len(precios_2026))

Sesiones 2021-2025: 5522990
Sesiones 2026: 1215238


## 2. Leer las cotizaciones y limpiar errores evidentes

In [2]:
# Limpiamos los datos de ambos periodos
for nombre, df in [("2021-2025", precios), ("2026", precios_2026)]:

    sesiones_antes = len(df)

    # Convertimos las fechas al horario de Nueva York
    df["fecha"] = (
        pd.to_datetime(df["timestamp"], utc=True)
        .dt.tz_convert("America/New_York")
        .dt.tz_localize(None)
        .dt.normalize()
    )

    # Ordenamos por empresa y fecha
    df = df.sort_values(["symbol", "fecha"])

    # Eliminamos sesiones duplicadas
    df = df.drop_duplicates(["symbol", "fecha"])

    # Eliminamos registros con precios o volúmenes incorrectos
    columnas_precios = ["open", "high", "low", "close", "vwap"]

    df = df[(df[columnas_precios] > 0).all(axis=1)]
    df = df[(df["volume"] > 0) & (df["high"] >= df["low"])].copy()

    df = df.reset_index(drop=True)

    print(nombre)
    print("Registros eliminados:", sesiones_antes - len(df))
    print("Empresas:", df["symbol"].nunique())
    print("Sesiones:", len(df))
    print()

    # Guardamos cada conjunto limpio por separado
    if nombre == "2026":
        precios_2026 = df
    else:
        precios = df

2021-2025
Registros eliminados: 134104
Empresas: 5419
Sesiones: 5388886

2026
Registros eliminados: 22941
Empresas: 5345
Sesiones: 1192297



## 3. Relacionar las empresas con la SEC

Cada empresa tiene un identificador único en la SEC, llamado **CIK**, que permite relacionar sus cotizaciones con la información publicada en sus informes oficiales, como el número de acciones en circulación.
Para ello, utilizamos un archivo que relaciona los símbolos bursátiles (*tickers*) con sus correspondientes CIK. Las empresas que no tienen un CIK identificado se excluyen del análisis porque no podemos consultar cuántas acciones tienen en circulación y, por tanto, no podemos calcular su capitalización bursátil ni comprobar si cumplen el criterio de entre 2 y 300 millones de dólares.

In [3]:
# Cargamos la relación entre los tickers y los identificadores de la SEC
mapa = pd.read_csv(DATOS / "mapa_ticker_cik.csv")

# Eliminamos los registros sin CIK y los tickers duplicados
mapa = mapa.dropna(subset=["cik"]).drop_duplicates("symbol")
mapa["cik"] = mapa["cik"].astype(int)

# Relacionamos las cotizaciones con la SEC
for nombre, df in [("2021-2025", precios), ("2026", precios_2026)]:

    empresas_antes = df["symbol"].nunique()
    df = df.merge(mapa[["symbol", "cik"]], on="symbol", how="inner")

    # Si una empresa aparece varias veces en un mismo día, conservamos el registro con mayor volumen
    df = df.sort_values("volume", ascending=False)
    df = df.drop_duplicates(["cik", "fecha"])
    df = df.sort_values(["symbol", "fecha"]).reset_index(drop=True)

    print(nombre)
    print("Empresas antes:", empresas_antes)
    print("Empresas identificadas:", df["cik"].nunique())
    print("Sesiones:", len(df))
    print()

    if nombre == "2026":
        precios_2026 = df
    else:
        precios = df

2021-2025
Empresas antes: 5419
Empresas identificadas: 5078
Sesiones: 5275114

2026
Empresas antes: 5345
Empresas identificadas: 5291
Sesiones: 1181824



## 4. Acciones en circulación según los informes publicados

Para seleccionar las empresas de entre 2 y 300 millones de dólares, necesitamos estimar su capitalización en cada sesión. Por eso utilizamos las acciones en circulación que las empresas han declarado a la SEC.

Los informes no se publican todos los días. Más adelante, para cada sesión, utilizaremos el último dato que ya se había publicado, sin recurrir a informes posteriores. Antes tenemos que ordenar estos datos y reducir los posibles errores.

### 4.1 Cargar y seleccionar los datos de acciones

El número de acciones puede aparecer en la portada del informe (dei) o en sus datos contables (us-gaap). Para no mezclar ambas fuentes, damos prioridad a dei y utilizamos us-gaap cuando una empresa no tiene registros dei. Esta prioridad se aplica por empresa, por lo que puede dejar fuera algunos datos históricos del balance.

In [4]:
# Cargamos los datos publicados en la SEC
sec = pd.read_csv(
    DATOS / "acciones_circulacion_sec.csv",
    parse_dates=["fin", "presentado"]
)

# Eliminamos registros sin un número válido de acciones
sec = sec[sec["acciones"] > 0].copy()

# Priorizamos las acciones indicadas en la portada del informe (dei).
# Si una empresa no tiene este dato, utilizamos el del balance (us-gaap).
cik_con_dei = sec.loc[
    sec["concepto"].str.startswith("dei"), "cik"
].unique()

sec = sec[
    sec["concepto"].str.startswith("dei") |
    ~sec["cik"].isin(cik_con_dei)
].copy()

print("Empresas con datos:", sec["cik"].nunique())

Empresas con datos: 4683


### 4.2 Ordenar los informes y eliminar duplicados

Un mismo dato puede aparecer repetido en varios informes. Agrupamos los registros de un mismo informe y evitamos conservar varias veces la misma fecha de referencia. También descartamos las fechas de referencia que retroceden con respecto a informes ya considerados.

Al sumar las acciones de un mismo informe asumimos que las cantidades corresponden a clases distintas de acciones. Es una aproximación que conviene tener presente.

In [5]:
# Sumamos las acciones declaradas en un mismo informe
sec = sec.groupby(
    ["cik", "accn", "fin"],
    as_index=False
).agg(
    presentado=("presentado", "max"),
    acciones=("acciones", "sum")
)

# Ordenamos por empresa y fecha de publicación
sec = sec.sort_values(["cik", "presentado", "fin"])

# Evitamos utilizar varias veces información de la misma fecha
sec = sec.drop_duplicates(["cik", "fin"], keep="first")

# Conservamos los informes cuyo periodo no retrocede
sec = sec[
    sec["fin"] >= sec.groupby("cik")["fin"].cummax()
]

# Si hay varios registros publicados el mismo día, conservamos el último
sec = sec.drop_duplicates(["cik", "presentado"], keep="last")

sec = sec.reset_index(drop=True)

print("Registros después de ordenar:", len(sec))

Registros después de ordenar: 165676


### 4.3 Detectar y tratar valores anómalos

Al comparar los informes de una misma empresa, algunas cifras de acciones cambian de forma extraordinaria. Pueden ser errores de los datos u operaciones reales de la empresa.

Aplicamos dos filtros para reducir el riesgo de calcular una capitalización incorrecta:

- Quitamos los **valores aislados** que se alejan mucho de los informes anterior y siguiente, cuando ambos muestran cifras parecidas.
- Excluimos las **empresas con cambios extremos** (más de 50 veces o menos de una milésima parte respecto al informe anterior), porque este procedimiento no puede interpretarlos con seguridad.

**Limitación:** estos filtros también pueden eliminar datos válidos. Los umbrales son decisiones prácticas de limpieza, no reglas de la SEC.

In [6]:
# Buscamos valores aislados: cambian mucho y luego vuelven a un nivel parecido
log_acciones = np.log(sec["acciones"])
anterior = log_acciones - log_acciones.groupby(sec["cik"]).shift(1)
siguiente = log_acciones - log_acciones.groupby(sec["cik"]).shift(-1)

valor_aislado = (
    (anterior.abs() > np.log(1.67)) &
    (siguiente.abs() > np.log(1.67)) &
    (np.sign(anterior) == np.sign(siguiente)) &
    ((anterior - siguiente).abs() < np.log(1.25))
)

print("Valores aislados eliminados:", valor_aislado.sum())
sec = sec[~valor_aislado].copy()

# Comparamos las acciones de cada informe con las del informe anterior
sec["cambio_acciones"] = (
    sec["acciones"] / sec.groupby("cik")["acciones"].shift(1)
)

# Excluimos empresas con cambios que este procedimiento no puede tratar bien
empresas_excluir = sec.loc[
    (sec["cambio_acciones"] < 0.001) |
    (sec["cambio_acciones"] > 50), "cik"
].unique()

print("Empresas excluidas:", len(empresas_excluir))
sec = sec[~sec["cik"].isin(empresas_excluir)].copy()
sec = sec.reset_index(drop=True)

# Volvemos a calcular los cambios después de la limpieza
sec["cambio_acciones"] = (
    sec["acciones"] / sec.groupby("cik")["acciones"].shift(1)
)

print("Aumentos superiores a 5 veces:", (sec["cambio_acciones"] > 5).sum())
print("Caídas superiores al 80 %:", (sec["cambio_acciones"] < 0.2).sum())

Valores aislados eliminados: 192
Empresas excluidas: 136
Aumentos superiores a 5 veces: 339
Caídas superiores al 80 %: 1134


### 4.4 Ajuste aproximado por splits

Los precios de Alpaca tienen en cuenta cambios como los *splits*, pero el número de acciones publicado por las empresas en la SEC no está ajustado de la misma manera. Por eso necesitamos adaptar ambos datos para poder calcular correctamente la capitalización bursátil.

Para hacerlo, identificamos posibles *splits* a partir de cambios importantes en el número de acciones. Sin embargo, este método es aproximado, ya que algunos cambios pueden deberse a otros motivos. Por tanto, la capitalización calculada será una estimación.

In [7]:
# Algunos cambios grandes pueden deberse a splits o contrasplits
ratio = sec["cambio_acciones"]
posible_split = (
    (ratio < 0.6) |
    ((ratio >= 1.9) & ((ratio - ratio.round()).abs() / ratio < 0.02))
)

# Si no parece un split, no aplicamos ningún cambio
sec["ratio_split"] = np.where(posible_split, ratio, 1.0)
sec["ratio_split"] = sec["ratio_split"].fillna(1.0)

# Ajustamos las acciones antiguas a la misma base que los precios de Alpaca
sec = sec.sort_values(["cik", "presentado"]).reset_index(drop=True)
sec["factor"] = (
    sec.iloc[::-1].groupby("cik")["ratio_split"].cumprod().iloc[::-1]
    / sec["ratio_split"]
)
sec["acciones_ajustadas"] = sec["acciones"] * sec["factor"]

# Resumen para revisar cuántos ajustes hemos estimado.
print("Posibles splits detectados:", (sec["ratio_split"] != 1).sum())
print("Empresas con posibles splits:", sec.loc[sec["ratio_split"] != 1, "cik"].nunique())
sec[["cik", "presentado", "acciones", "ratio_split", "acciones_ajustadas"]].head()

Posibles splits detectados: 2150
Empresas con posibles splits: 1219


,cik,presentado,acciones,ratio_split,acciones_ajustadas
0,1750,2010-09-23,39662816.0,1.0,39662816.0
1,1750,2010-12-21,39677152.0,1.0,39677152.0
2,1750,2011-03-22,39739031.0,1.0,39739031.0
3,1750,2011-07-13,39682142.0,1.0,39682142.0
4,1750,2011-09-23,40460631.0,1.0,40460631.0


## 5. Capitalización de cada día

Para calcular la capitalización de una empresa necesitamos su precio y el número de acciones en circulación. Asociamos cada sesión al último dato de acciones publicado en la SEC, sin utilizar informes futuros.

Como solo disponemos de la **fecha**, y no de la hora exacta de publicación, consideramos que un informe empieza a estar disponible **al día siguiente**. Así evitamos usar información que pudiera haberse publicado después del cierre. También descartamos datos de acciones con más de 400 días, porque podrían estar desactualizados.

Por último, multiplicamos el cierre por las acciones ajustadas para estimar la capitalización. Esta cifra sigue siendo aproximada por los posibles errores al identificar splits.

In [8]:
# Sin hora de publicación, utilizamos cada informe desde el día siguiente.
sec["disponible_desde"] = sec["presentado"] + pd.Timedelta(days=1)

for nombre, df in [("2021-2025", precios), ("2026", precios_2026)]:

    # Relacionamos cada sesión con la información que ya estaba disponible.
    df = pd.merge_asof(
        df.sort_values("fecha"),
        sec[["cik", "presentado", "disponible_desde", "acciones",
             "acciones_ajustadas", "factor"]].sort_values("disponible_desde"),
        left_on="fecha",
        right_on="disponible_desde",
        by="cik",
        direction="backward"
    )

    # No utilizamos datos de acciones con más de 400 días de antigüedad.
    antiguo = (df["fecha"] - df["presentado"]).dt.days > 400
    df.loc[antiguo, ["acciones", "acciones_ajustadas", "factor"]] = np.nan

    # Calculamos una capitalización aproximada usando el cierre ajustado.
    df["capitalizacion"] = df["close"] * df["acciones_ajustadas"]
    df["precio_nominal"] = df["close"] * df["factor"]

    df = df.sort_values(["symbol", "fecha"]).reset_index(drop=True)

    print(nombre, "- Sesiones con capitalización:", df["capitalizacion"].notna().sum())
    print("  Empresas con capitalización:", df.loc[df["capitalizacion"].notna(), "symbol"].nunique())

    if nombre == "2026":
        precios_2026 = df
    else:
        precios = df

2021-2025 - Sesiones con capitalización: 4210093
  Empresas con capitalización: 4181
2026 - Sesiones con capitalización: 934129
  Empresas con capitalización: 4189


## 6. Variables de mercado

A partir de los datos de mercado, calculamos variables que nos permiten analizar cómo ha evolucionado cada acción en cuanto a precio, volumen y volatilidad.

Los cálculos se realizan por empresa, utilizando tanto los datos de la sesión actual como los de sesiones anteriores. De esta manera, podemos estudiar su comportamiento reciente y utilizar esta información para intentar predecir si el precio subirá o bajará en la siguiente sesión.

Para ello, utilizamos `shift()` para consultar valores de sesiones anteriores y `rolling()` para realizar cálculos sobre varios días consecutivos.

In [9]:
for nombre, df in [("2021-2025", precios), ("2026", precios_2026)]:

    df = df.sort_values(["symbol", "fecha"]).copy()
    g = df.groupby("symbol")

    # Variaciones del precio
    df["rent_1d"] = df["close"] / g["close"].shift(1) - 1
    df["gap"] = df["open"] / g["close"].shift(1) - 1
    df["rent_intradia"] = df["close"] / df["open"] - 1
    df["rango"] = (df["high"] - df["low"]) / df["close"]

    df["pos_cierre"] = (
        (df["close"] - df["low"]) /
        (df["high"] - df["low"]).replace(0, np.nan)
    ).fillna(0.5)

    df["dist_vwap"] = df["close"] / df["vwap"] - 1

    # Rentabilidad y volatilidad de las últimas sesiones
    for dias in [5, 10, 20]:
        df[f"rent_{dias}d"] = df["close"] / g["close"].shift(dias) - 1
        df[f"volat_{dias}d"] = g["rent_1d"].transform(
            lambda x: x.rolling(dias).std()
        )

    df["dist_max_20d"] = (
        df["close"] /
        g["high"].transform(lambda x: x.rolling(20).max()) - 1
    )

    # Volumen y número de operaciones
    media_volumen_20 = g["volume"].transform(
        lambda x: x.shift(1).rolling(20).mean()
    )

    media_operaciones_20 = g["trade_count"].transform(
        lambda x: x.shift(1).rolling(20).mean()
    )

    df["vol_rel_1d"] = np.log(df["volume"] / g["volume"].shift(1))
    df["vol_rel_5d"] = np.log(
        g["volume"].transform(lambda x: x.rolling(5).mean()) /
        media_volumen_20
    )
    df["vol_rel_20d"] = np.log(df["volume"] / media_volumen_20)
    df["ops_rel_20d"] = np.log(df["trade_count"] / media_operaciones_20)
    df["log_volumen_dolares"] = np.log(df["close"] * df["volume"])

    # Tamaño de la empresa
    df["rotacion"] = df["volume"] / df["acciones_ajustadas"]
    df["log_capitalizacion"] = np.log(df["capitalizacion"])
    df["log_precio"] = np.log(df["precio_nominal"])

    # Resultado que queremos predecir: si sube en la siguiente sesión
    df["variacion_siguiente"] = (
        g["close"].shift(-1) / df["close"] - 1
    ) * 100

    df["sube"] = np.where(
        df["variacion_siguiente"].isna(),
        np.nan,
        (df["variacion_siguiente"] > 0).astype(int)
    )

    # Número de sesiones disponibles para cada empresa
    df["n_sesion"] = g.cumcount()

    if nombre == "2026":
        precios_2026 = df
    else:
        precios = df

print("Variables calculadas para ambos periodos")

Variables calculadas para ambos periodos


## 7. Preparar la muestra general de small caps
Para el modelo general queremos predecir si una acción subirá en la sesión siguiente. Por eso usamos las variables calculadas al cierre de cada día.

Nos quedamos con las sesiones cuya capitalización estimada está entre 2 y 300 millones de dólares, que tienen al menos 20 sesiones de historial y no presentan valores necesarios sin calcular. La muestra general utiliza 2021–2025: 2021–2023 para entrenamiento, 2024 para validación y 2025 para prueba. 2026 no entra en este modelo general.

In [10]:
# Variables que utilizará el modelo general
VARIABLES = [
    "rent_1d", "gap", "rent_intradia", "rango", "pos_cierre", "dist_vwap",
    "rent_5d", "rent_10d", "rent_20d", "volat_5d", "volat_10d", "volat_20d",
    "dist_max_20d", "vol_rel_1d", "vol_rel_5d", "vol_rel_20d", "ops_rel_20d",
    "log_volumen_dolares", "rotacion", "log_capitalizacion", "log_precio"
]

# Seleccionamos las sesiones que cumplen las condiciones del estudio
print("Sesiones iniciales:", len(precios))

muestra = precios[precios["capitalizacion"].between(2e6, 300e6)].copy()
print("Dentro del rango de capitalización:", len(muestra))

muestra = muestra[muestra["n_sesion"] >= 20]
print("Con al menos 20 sesiones de historial:", len(muestra))

muestra = muestra.replace([np.inf, -np.inf], np.nan)

antes = len(muestra)
muestra = muestra.dropna(subset=VARIABLES + ["variacion_siguiente"])

print("Descartadas por datos incompletos:", antes - len(muestra))
print("Sesiones finales:", len(muestra))

# Dividimos los datos por años, sin mezclarlos de forma aleatoria
muestra["anio"] = muestra["fecha"].dt.year
muestra["conjunto"] = np.select(
    [muestra["anio"] <= 2023, muestra["anio"] == 2024],
    ["entrenamiento", "validacion"], default="prueba"
)
muestra["segmento"] = np.where(
    muestra["capitalizacion"] < 50e6, "nano cap (2-50 M$)", "micro cap (50-300 M$)"
)

# Añadimos el nombre de la empresa y la bolsa en la que cotiza
universo = pd.read_csv(DATOS / "universo.csv")
muestra = muestra.merge(
    universo[["symbol", "name", "exchange", "status"]].drop_duplicates("symbol"),
    on="symbol", how="left"
)

muestra.to_parquet(DATOS / "muestra_smallcaps.parquet", index=False)
print("Empresas en la muestra:", muestra["symbol"].nunique())
print("Sesiones por periodo:")
print(muestra.groupby("conjunto").size())

Sesiones iniciales: 5275114
Dentro del rango de capitalización: 1266268
Con al menos 20 sesiones de historial: 1250207
Descartadas por datos incompletos: 1360
Sesiones finales: 1248847
Empresas en la muestra: 2014
Sesiones por periodo:
conjunto
entrenamiento    631169
prueba           325091
validacion       292587
dtype: int64


## 8. Preparar las sesiones de apertura y los gaps

Para estudiar el patrón gap and crap, queremos saber si una acción que abre con una subida del 40 % o más termina cerrando por debajo de su precio de apertura. A diferencia del modelo general, aquí la predicción se hace al abrir el mercado. Por eso utilizamos el gap de apertura y las variables del día anterior, evitando utilizar información que solo conoceríamos al cierre.

Preparamos por separado 2021–2025 y 2026. El archivo de 2026 incluye algunas sesiones de finales de 2025 para calcular el historial, pero solo conservaremos eventos de 2026 hasta el 12 de octubre, tal como se establece en la memoria del TFM.

In [11]:
# Variables que necesitamos conocer antes de la apertura
previas = [
    "capitalizacion", "rent_1d", "rent_5d", "rent_20d", "volat_20d",
    "dist_max_20d", "pos_cierre", "vol_rel_20d", "ops_rel_20d",
    "log_volumen_dolares", "rotacion", "gap"
]

VARIABLES_GAP = [
    "gap", "log_precio_apertura", "log_cap_previa", "rent_1d_previo", "rent_5d_previo",
    "rent_20d_previo", "volat_20d_previo", "dist_max_20d_previo", "pos_cierre_previo",
    "vol_rel_20d_previo", "ops_rel_20d_previo", "log_volumen_dolares_previo",
    "rotacion_previo", "gap_previo"
]

lista_eventos = []

for nombre, df in [("2021-2025", precios), ("2026", precios_2026)]:

    df = df.sort_values(["symbol", "fecha"]).copy()

    # Desplazamos las variables al día anterior
    for variable in previas:
        df[variable + "_previo"] = df.groupby("symbol")[variable].shift(1)

    # La empresa debía ser una small cap al cierre anterior
    apertura = df[df["capitalizacion_previo"].between(2e6, 300e6)].copy()
    apertura = apertura[apertura["n_sesion"] >= 20]
    apertura = apertura.replace([np.inf, -np.inf], np.nan)
    apertura = apertura.dropna(subset=[
        "gap", "capitalizacion_previo", "rent_1d_previo",
        "vol_rel_20d_previo", "volat_20d_previo"
    ])

    if nombre == "2026":
        # Las sesiones de finales de 2025 solo servían para calcular el historial
        apertura = apertura[apertura["fecha"].between("2026-01-01", "2026-10-12")].copy()
        apertura["conjunto"] = "prueba_final"
    else:
        apertura["anio"] = apertura["fecha"].dt.year
        apertura["conjunto"] = np.select(
            [apertura["anio"] <= 2023, apertura["anio"] == 2024],
            ["entrenamiento", "validacion"], default="prueba"
        )

    apertura["anio"] = apertura["fecha"].dt.year

    # Guardamos estas sesiones para comparar los patrones posteriormente
    columnas_sesiones = [
        "fecha", "anio", "conjunto", "gap", "rent_intradia",
        "rent_1d", "vol_rel_20d", "variacion_siguiente"
    ]
    archivo = "sesiones_apertura_2026.parquet" if nombre == "2026" else "sesiones_apertura.parquet"
    apertura[columnas_sesiones].to_parquet(DATOS / archivo, index=False)

    # Seleccionamos los gaps alcistas del 40 % o más
    gaps = apertura[apertura["gap"] >= 0.40].copy()
    gaps["precio_apertura"] = gaps["open"] * gaps["precio_nominal"] / gaps["close"]
    gaps["log_precio_apertura"] = np.log(gaps["precio_apertura"])
    gaps["log_cap_previa"] = np.log(gaps["capitalizacion_previo"])
    gaps["crap"] = (gaps["close"] < gaps["open"]).astype(int)
    gaps["id_evento"] = gaps["symbol"] + "_" + gaps["fecha"].dt.strftime("%Y-%m-%d")
    gaps = gaps.dropna(subset=VARIABLES_GAP)

    lista_eventos.append(gaps)
    print(nombre, "- Gaps encontrados:", len(gaps))

2021-2025 - Gaps encontrados: 1307
2026 - Gaps encontrados: 511


## 9. Guardar los eventos y comprobar los resultados

Guardamos los gaps de ambos periodos en un solo archivo, manteniendo la columna `conjunto` para distinguir entrenamiento, validación, prueba y prueba final de 2026. Al terminar, comprobamos cuántos eventos hay en cada periodo y que las fechas de publicación de los informes utilizados sean anteriores a las sesiones.

In [12]:
# Juntamos los gaps de ambos periodos, sin mezclar sus etiquetas temporales
columnas_eventos = [
    "id_evento", "symbol", "cik", "fecha", "anio", "conjunto",
    "open", "close", "precio_apertura", "crap"
] + VARIABLES_GAP

eventos = pd.concat(lista_eventos, ignore_index=True)
eventos = eventos[columnas_eventos].sort_values("fecha").reset_index(drop=True)
eventos.to_csv(DATOS / "eventos_gap40.csv", index=False)

print("Sesiones del modelo general:", len(muestra))
print("Gaps totales:", len(eventos))
print("Gaps por periodo:")
print(eventos.groupby("conjunto")["crap"].agg(["size", "mean"]))

# Comprobamos que no se hayan utilizado informes futuros
for nombre, df in [("2021-2025", precios), ("2026", precios_2026)]:
    con_informe = df.dropna(subset=["presentado"])
    informes_no_disponibles = (con_informe["disponible_desde"] > con_informe["fecha"]).sum()
    print(nombre, "- Sesiones con informes no disponibles:", informes_no_disponibles)

Sesiones del modelo general: 1248847
Gaps totales: 1818
Gaps por periodo:
               size      mean
conjunto                     
entrenamiento   421  0.726841
prueba          517  0.709865
prueba_final    511  0.702544
validacion      369  0.672087
2021-2025 - Sesiones con informes no disponibles: 0
2026 - Sesiones con informes no disponibles: 0


## Archivos preparados para los siguientes notebooks
- muestra_smallcaps.parquet: sesiones de 2021–2025 para el modelo general.
- sesiones_apertura.parquet: sesiones de 2021–2025 para comparar los cinco patrones.
- sesiones_apertura_2026.parquet: sesiones de 2026 para comprobar los patrones fuera del periodo de selección.
- eventos_gap40.csv: gaps de 2021–2026, con su periodo identificado, para el modelo de gaps y el análisis de noticias.
La descarga de datos se realiza aparte. Los precios y las acciones en circulación se han preparado con las aproximaciones descritas en los apartados anteriores; especialmente, el ajuste por splits no equivale a una reconstrucción exacta de todas las operaciones corporativas.